# L02 — Chemical Reaction Networks I

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lab-biotek-bio-ugm/TKBM262615_Practicals/blob/main/notebooks/02_mass_action.ipynb)

**Course:** TBM263214 Komputasi Biologi (Computational Biology) · **Week 2** · **CPMK1**

Companion notebook for [`scripts/02_mass_action.py`](../scripts/02_mass_action.py), based on the course lecture note [`02-chemical-reaction-networks-1-en.md`](https://github.com/lab-biotek-bio-ugm/TKBM262615/blob/main/lecture-notes/02-chemical-reaction-networks-1-en.md).

Simulates the Law of Mass Action: a reversible reaction A ⇌ B and a dimerization reaction 2R ⇌ R2, including mass-conservation checks.


## Learning Objectives

After this notebook, you should be able to:
1. **Apply the Law of Mass Action** to write ODEs for simple chemical reactions. *(CPMK1)*
2. **Construct models** of zeroth-order, first-order, and second-order reactions. *(CPMK1)*
3. **Write reversible reaction models** with forward rate constant $k_+$ and reverse rate constant $k_-$. *(CPMK1)*
4. **Determine** the order of a reaction from the form of the ODE and **solve** first-order models. *(CPMK1)*


In [ ]:
# Setup — Colab already ships numpy, scipy, and matplotlib, so no pip install is needed.
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


## Key Concepts

- **Law of Mass Action:** the rate of a chemical reaction is proportional to the product of reactant concentrations raised to the power of their stoichiometric coefficients.
- **Reaction order:** zeroth order ($\frac{dX}{dt}=k$), first order ($\frac{dX}{dt}=-kX$), second order ($\frac{dX}{dt}=-kXY$ or $-kX^2$).
- **Reversible reaction** $A \rightleftharpoons B$: forward constant $k_+$, reverse constant $k_-$; *steady state* when $k_+[A] = k_-[B]$, giving equilibrium ratio $[B]/[A] = k_+/k_-$.
- **One-molecule concentration:** in a small cell (~1 fL), one molecule ≈ 1 nM — the lower bound of biologically relevant concentration (*Swain §2.1.4*).
- **Diffusion limit:** the bimolecular (macroscopic) rate constant cannot exceed the diffusion limit (~$10^9\,\mathrm{M^{-1}s^{-1}}$) (*Swain §2.1.3*).

## Key Equations

- First-order decay:
$$ \frac{dX}{dt} = -kX \quad\Rightarrow\quad X(t) = X_0 e^{-kt}, \qquad t_{1/2} = \frac{\ln 2}{k} $$

- Reversible $A \xrightleftharpoons[k_-]{k_+} B$:
$$ \frac{dA}{dt} = -k_+ A + k_- B, \qquad \frac{dB}{dt} = k_+ A - k_- B, \qquad A+B = A_0+B_0 \ (\text{conserved}) $$

- Dimerization $2R \rightleftharpoons R_2$:
$$ \frac{dR}{dt} = -2f R^2 + 2b R_2, \qquad \frac{dR_2}{dt} = f R^2 - b R_2, \qquad [R]+2[R_2] = \text{constant} $$


In [ ]:
# Reversible reaction A <-> B
def reversible(t, y, kp, km):
    A, B = y
    dA = -kp * A + km * B
    dB = kp * A - km * B
    return [dA, dB]

# Dimerization 2R <-> R2 (note the factor of 2 on R)
def dimerisation(t, y, f, b):
    R, R2 = y
    dR = -2 * f * R**2 + 2 * b * R2
    dR2 = f * R**2 - b * R2
    return [dR, dR2]


In [ ]:
kp, km = 0.8, 0.2          # forward/reverse rate constants (1/s)
A0, B0 = 1.0, 0.0
t_eval = np.linspace(0, 10, 400)

sol = solve_ivp(reversible, (0, 10), [A0, B0], t_eval=t_eval, args=(kp, km))
A, B = sol.y
print("Reversible A <-> B:")
print(f"  Steady state B/A = kp/km = {kp/km:.2f}  |  simulated = {B[-1]/A[-1]:.2f}")
print(f"  Mass conservation: A+B = {A0+B0:.2f} -> {A[-1]+B[-1]:.2f}")


In [ ]:
f, b = 0.5, 0.1
R0, R20 = 1.0, 0.0
sol2 = solve_ivp(dimerisation, (0, 10), [R0, R20], t_eval=t_eval, args=(f, b))
R, R2 = sol2.y

# Conservation: [R] + 2[R2] = constant
conserved = R + 2 * R2
print("Dimerization 2R <-> R2:")
print(f"  Conservation [R]+2[R2]: start {R0+2*R20:.2f} -> end {conserved[-1]:.2f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(sol.t, A, label="A")
axes[0].plot(sol.t, B, label="B")
axes[0].set_title("Reversible A <-> B")
axes[0].legend()
axes[1].plot(sol2.t, R, label="R")
axes[1].plot(sol2.t, R2, label="R2")
axes[1].set_title("Dimerization 2R <-> R2")
axes[1].legend()
for ax in axes:
    ax.set_xlabel("time t")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Python Exercises

### Exercise 1 *(CPMK1)*
Simulate the reversible reaction with ratios $k_+/k_- = 10$ and $0.1$; compare the $B/A$ ratio at *steady state*.


In [ ]:
# TODO: Exercise 1 — resimulate `reversible` with kp/km = 10 and kp/km = 0.1, compare B/A at steady state


### Exercise 2 *(CPMK1)*
Verify the conservation $[R] + 2[R_2] = \text{constant}$ in the dimerization simulation above.


In [ ]:
# TODO: Exercise 2 — check R + 2*R2 stays constant across the whole time series (not just start/end)


### Exercise 3 *(CPMK1)*
*One-molecule concentration:* in a cell volume $V = 1$ fL, compute the concentration of one molecule ($1/(N_A V)$). Interpret the result (*Swain §2.1.4*).


In [ ]:
# TODO: Exercise 3 — compute 1 / (N_A * V) for V = 1e-15 L (1 fL), N_A = 6.022e23 /mol


## Discussion Questions

1. Why do second-order reactions have a different "saturation effect" from first-order ones?
2. How does the law of mass action apply to enzymatic reactions that involve complex steps?
3. What are the consequences if the rate constant has the wrong units?

## Reading

- Ingalls (2013) Chapter 2: chemical reaction kinetics, Law of Mass Action, conservation.
- Alon (2006) Section 1.2: reaction kinetics basics.
- Strogatz (2014) Chapter 2: first-order ODE solutions & stability.
- Swain, PSB notes §2.1–2.3 (chemical rate equations, equilibrium & detailed balance, law of mass action), §2.1.1 (dimerisation), §2.1.3–2.1.4 (diffusion-limited reactions, concentration of one molecule). [notes.pdf](https://swainlab.bio.ed.ac.uk/psb/lectures/notes.pdf) · [course site](https://swainlab.bio.ed.ac.uk/psb/index.html)
